In [1]:
#Imports
import numpy as np
import h5py

## Section 0, user input

In [2]:
#Edit these variables, all the rest is automatic.
filename="SRP_CNR-IMM@CT_input.ASC"
output="SRP_CNR-IMM@CT_output.nxs"

#SAMPLE NAME AND IDENTIFIER
#OPTIONAL UNLESS FOR NFFA-DI
sample_name='Example name'
sample_identifier='Example ID'

#COMMENT
#OPTIONAL. If left blank, the measurement file's comment section will be used 
#(if it's not blank itself)
comment=''

## Section 1, read SRP file

In [3]:


def read_header(file_handle):
    """
    Reads the header rows from the file and returns a dictionary with metadata.

    Note that all rows are read as strings, even numerical ones. 
    Eventual conversion will happen at nexus file writing.
    """
    header_keys = [
        "title", "sample_structure", "lapping_paste", "crystallographic_orientation",
        "experiment_identifier", "spreading_experiment_type", "operator_name", "tip_step",
        "spatial_resolution", "instrument_name", "experiment_location", "date", "time",
        "applied_force", "intertip_distance", "measurment_identifier"
    ]
    #unused in the script: operator_name, tip_step, spatial_resolution, measurement_identifier
    
    if file_handle.readline().strip() != "\"{*HEADER*}\",21":
        raise ValueError("Format invalid. Header section row does not match expected string.")
    
    header = {}
    
    for i, key in enumerate(header_keys, start=1):
        header[key] = file_handle.readline().replace("\"", "").strip()
    
    # Skip unused rows (17 through 22)
    for _ in range(22-17):
        file_handle.readline()
    
    return header

def read_comment(file_handle):
    """
    Comment rows reading.

    Comment is read as a single string, including newline characters.
    """
    
    #check that the line declaring comments is of the expected form, and read how many lines are in it.
    #purposefully rigid design!
    commentrow=file_handle.readline().strip()
    if commentrow[0:14] != "\"{*COMMENT*}\",":
        raise ValueError("Format invalid. Comment section row does not match expected string")
    num_comment_lines=int(commentrow[14:])

    comment=""
    
    for i in range(num_comment_lines):
        comment+=file_handle.readline().replace("\"", "") #read each line, removing quotation marks only
    
    return comment

def read_layers(file_handle):
    """
    Reads the layer rows from the file and returns a list of dictionaries.
    Note that all parameters of a layer are read as strings. 
    Conversion will happen at nexus file writing.
    """
    
    layer_keys = [
        "layer_index", "firstpoint", "endpoint", "PorN", "manual", "thickness",
        "method", "junction_depth", "layer_resistance", "active_dose"
    ]

    #check that the line declaring layers is of the expected form, and read how many lines/layers are in it.
    #purposefully rigid design!
    layerrow=file_handle.readline().strip()
    if layerrow[0:20] != "\"{*LAYER SUMMARY*}\",":
        raise ValueError("Format invalid. Layer section row does not match expected string")
    num_layer_lines=int(layerrow[20:])

    layers=[]
    
    for i in range(num_layer_lines):
        layer=file_handle.readline().strip().replace("\"", "")
        layer_params=layer.split(",")
        layer_dict=dict(zip(layer_keys, layer_params))
        layers.append(layer_dict)
    
    return layers

def read_measurement(file_handle):
    """
    Reads the measurement rows from the file and returns a ndarray named data, whence
    data[:,0] is the first column (index)
    data[:,1] is the second column (depth)
    data[:,2] is the third column (spreading resistance)
    data[:,3] is the fourth column (resistivity)
    data[:,4] is the fifth column (carrier concentration)
    """

    #check that the line declaring measurement is of the expected form, and read how many lines are in it.
    #purposefully rigid design!
    measurementrow=file_handle.readline().strip()
    if measurementrow[0:25] != "\"{*MEASUREMENT VALUES*}\",":
        raise ValueError("Format invalid. Measurement section row does not match expected string")
    num_measurement_lines=int(measurementrow[25:])

    data_rows = []

    for i in range(num_measurement_lines):
        data_rows.append(file_handle.readline().strip().split(","))
    
    data = np.array(data_rows, dtype=float)

    return data

def read_calibration(file_handle):
    """
    Reads one block of calibration data and returns a dictionary "calibration" composed of:
    calibration['type'] = P or N, the type of layer
    calibration['tip_contact_radius']
    calibration['tolerance']
    calibration['data'], a ndarray whence
    calibration['data'][:,0] is the first column (index)
    calibration['data'][:,1] is the second column (spreading resistance)
    calibration['data'][:,2] is the third column (resistivity)
    """

    calibration={}

    #check that the line declaring calibration is of the expected form, and read how many lines are in it.
    #purposefully rigid design!
    calibrationrow=file_handle.readline().strip()
    if calibrationrow[0:23] != "\"{*CALIBRATION DATA*}\",":
        raise ValueError("Format invalid. Calibration section row does not match expected string")
    lineinfo=calibrationrow[23:].split(",")
    
    calibration['type']=lineinfo[0]

    num_calibration_lines=int(lineinfo[1])

    cal_metadata=file_handle.readline().strip().split(",")
    calibration['tip_contact_radius']=cal_metadata[0]
    calibration['tolerance']=cal_metadata[1]

    calibration_rows = []

    for i in range(num_calibration_lines):
        calibration_rows.append(file_handle.readline().strip().split(","))
    
    calibration['data'] = np.array(calibration_rows, dtype=float)

    return calibration

try: f.close() #close the file if it was already open, useful for debug and reruns.
except NameError: pass

file=open(filename)

header = read_header(file)
file_comment = read_comment(file)
if comment == '':
    comment=file_comment
layers = read_layers(file)
data=read_measurement(file)
calibration1=read_calibration(file)
calibration2=read_calibration(file)

file.close()

## Section 2, write into a NeXus file

In [4]:

try: f.close() #close the output file if it was already open, useful for debug/reruns.
except NameError: pass

f=h5py.File(output, 'w')

f.create_group('/entry')
f['/entry'].attrs['NX_class']='NXentry'
f['/entry'].attrs['default']='data'

f["/entry/definition"]="NXspreading"
f["/entry/definition"].attrs["version"]="2025-12-02"
#f["/entry/definition"].attrs["url"]="insert github permalink here"

f["/entry/comment"] = comment

#Intendedly returns an error if the key is not present;
#Create and assign nexus file fields if values are not empty.
if header['title']: 
    f["/entry/title"]=header['title']

# Add spreading experiment metadata
if header['spreading_experiment_type'].strip()=='2 pt probe - I':
    f["/entry/spreading_experiment_type"] = '2 probe'
else:
    raise ValueError('expected 2 pt probe, check measurement file and update script')
if header['experiment_identifier']:
    f["/entry/experiment_identifier"] = header['experiment_identifier']
if header['experiment_location']:
    if header['experiment_location'] == "CNR-IMM":
        f["/entry/experiment_location"] = "CNR - IMM Catania"
    else:
        f["/entry/experiment_location"] = header['experiment_location']

# Convert header['date'] to yyyy-mm-dd format
if header['date']:
    date_parts = header['date'].split(" ")
    if len(date_parts) == 3:
        year = "20" + date_parts[2]
        month = {
            "Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
            "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
            "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"
        }.get(date_parts[1], "01")
        day = date_parts[0].zfill(2)
        header['date'] = f"{year}-{month}-{day}"

if header['date']:
    if header['time']:
        f["/entry/start_time"] = f"{header['date']}T{header['time']}"
    else:
        f["/entry/start_time"] = header['date']

# Add instrument group
f.create_group('/entry/instrument')
f['/entry/instrument'].attrs['NX_class'] = 'NXinstrument'
f["/entry/instrument/name"] = header['instrument_name']
if header['applied_force']:
    f["/entry/instrument/applied_force"] = float(header['applied_force'])
    f["/entry/instrument/applied_force"].attrs['units'] = 'g'
if header['intertip_distance']:
    f["/entry/instrument/tip_distance"] = float(header['intertip_distance'])
    f["/entry/instrument/tip_distance"].attrs['units'] = 'um'

# Add sample group
f.create_group('/entry/sample')
f['/entry/sample'].attrs['NX_class'] = 'NXsample'
#TO DO, ask in user input section as optional?
if sample_name:
    f["/entry/sample/name"] = sample_name
if sample_identifier:
    f["/entry/sample/identifier"] = sample_identifier
if header['sample_structure']:
    f["/entry/sample/sample_structure"] = header['sample_structure']
#link to data/process/layer_model at the end

#assuming sample preparation is always lapping, otherwise make a different conditional
if True:
    f["/entry/sample/surface_preparation_method"] = "lapping" #assumed default
    if header['lapping_paste']:
        f["/entry/sample/lapping_paste"] = header['lapping_paste']

if header['crystallographic_orientation']:
    f["/entry/sample/crystallographic_orientation"] = header['crystallographic_orientation']
    if True: #Assuming Miller indices notation
        f["/entry/sample/crystallographic_orientation"].attrs['notation']='Miller'

# Add data group
f.create_group('/entry/data')
f['/entry/data'].attrs['NX_class'] = 'NXdata'
f['/entry/data'].attrs['axes'] = 'depth'

# Check for available data columns
has_spre_data = not np.all(data[:, 2] == 0)
has_res_data = not np.all(data[:, 3] == 0)
has_car_data = not np.all(data[:, 4] == 0)

# Set data_type attribute
data_types = []
if has_spre_data:
    data_types.append("spreading resistance")
if has_res_data:
    data_types.append("resistivity")
if has_car_data:
    data_types.append("carrier density")
f['/entry/data/data_type'] = data_types if len(data_types) > 1 else data_types[0]

# Add depth, and conditionally data fields
f["/entry/data/depth"] = data[:, 1]
f["/entry/data/depth"].attrs['units'] = 'um'

if has_spre_data:
    f["/entry/data/spreading_resistance"] = 10 ** data[:, 2]
    f["/entry/data/spreading_resistance"].attrs['units'] = 'Ohm'
if has_res_data:
    f["/entry/data/resistivity"] = 10 ** data[:, 3]
    f["/entry/data/resistivity"].attrs['units'] = 'Ohm*cm'
if has_car_data:
    f["/entry/data/carrier_concentration"] = 10 ** data[:, 4]
    f["/entry/data/carrier_concentration"].attrs['units'] = '1/cm^3'

# Set signal and auxiliary axes attributes
f['/entry/data'].attrs['SILX_style'] = "{\"signal_scale_type\":\"log\"}"
if has_spre_data:
    f['/entry/data'].attrs['signal'] = 'spreading_resistance'
    has_spre_data = False #don't write in auxiliary axes if it's the signal
elif has_res_data:
    f['/entry/data'].attrs['signal'] = 'resistivity'
    has_res_data = False
elif has_car_data:
    f['/entry/data'].attrs['signal'] = 'carrier_concentration'
    has_car_data = False

auxiliary_signals = []
if has_spre_data: #should never happen
    auxiliary_signals.append('spreading_resistance')
if has_res_data:
    auxiliary_signals.append('resistivity')
if has_car_data:
    auxiliary_signals.append('carrier_concentration')
if auxiliary_signals:
    f['/entry/data'].attrs['auxiliary_signals'] = auxiliary_signals if len(auxiliary_signals) > 1 else auxiliary_signals[0]

# Add process group
f.create_group('/entry/data/process')
f['/entry/data/process'].attrs['NX_class'] = 'NXprocess'

# Add note group
f.create_group('/entry/data/process/note')
f['/entry/data/process/note'].attrs['NX_class'] = 'NXnote'
f["/entry/data/process/note/description"] = "Interpolation to calibration data based on each layer's type."

# Add layer model group
f.create_group('/entry/data/process/layer_model')
f['/entry/data/process/layer_model'].attrs['NX_class'] = 'NXobject'
f["/entry/data/process/layer_model/number_of_layers"] = len(layers)

# Add layers procedurally
for i, layer in enumerate(layers):
    layer_group = f"/entry/data/process/layer_model/layer{i + 1}"
    f.create_group(layer_group)
    f[layer_group].attrs['NX_class'] = 'NXobject'
    f[f"{layer_group}/index"] = int(layer['layer_index'])
    f[f"{layer_group}/starting_point"] = float(layer['firstpoint'])
    f[f"{layer_group}/ending_point"] = float(layer['endpoint'])
    f[f"{layer_group}/type"] = layer['PorN']
    f[f"{layer_group}/type"].attrs['mode'] = layer['manual']
    f[f"{layer_group}/thickness"] = float(layer['thickness'])
    f[f"{layer_group}/thickness"].attrs['units'] = 'um'
    f[f"{layer_group}/method"] = layer['method']
    f[f"{layer_group}/junction_depth"] = float(layer['junction_depth'])
    f[f"{layer_group}/junction_depth"].attrs['units'] = 'um'
    f[f"{layer_group}/layer_resistance"] = float(layer['layer_resistance'])
    f[f"{layer_group}/layer_resistance"].attrs['units'] = "Ohm/square"
    f[f"{layer_group}/active_dose"] = float(layer['active_dose'])
    f[f"{layer_group}/active_dose"].attrs['units'] = '1/cm^2'

# Determine which calibration corresponds to P or N type
calibrations = [calibration1, calibration2]
for calibration in calibrations:
    if calibration['type'] == 'P':
        group_name = '/entry/data/process/P_calibration_data'
    elif calibration['type'] == 'N':
        group_name = '/entry/data/process/N_calibration_data'
    else:
        raise ValueError(f"Unexpected calibration type: {calibration['type']}")

    # Create the group
    f.create_group(group_name)
    f[group_name].attrs['NX_class'] = 'NXobject'

    # Add calibration metadata
    f[f"{group_name}/tip_contact_radius"] = float(calibration['tip_contact_radius'])
    f[f"{group_name}/tip_contact_radius"].attrs['units'] = 'um'

    f[f"{group_name}/tolerance"] = float(calibration['tolerance'])
    f[f"{group_name}/tolerance"].attrs['units'] = 'percentage'

    # Add calibration data
    f[f"{group_name}/spreading_resistance"] = 10 ** calibration['data'][:, 1]
    f[f"{group_name}/spreading_resistance"].attrs['units'] = 'Ohm'

    f[f"{group_name}/calibrated_resistivity"] = 10 ** calibration['data'][:, 2]
    f[f"{group_name}/calibrated_resistivity"].attrs['units'] = 'Ohm*cm'
    f[f"{group_name}/calibrated_resistivity"].attrs['certified'] = True

#LINKS
f['/entry/data/process/layer_model'].attrs['target']='/entry/data/process/layer_model'
f['/entry/sample/layer_model'] = f['/entry/data/process/layer_model']

#close file
f.close()